# Conformal Symmetry of the 2D Laplace and Liouville Equations

This tutorial demonstrates using **`symlie`** to analyze the conformal geometry and symmetry groups of classical elliptic and hyperbolic equations, following **F. Güngör** (*Lie symmetry group methods for differential equations*, arXiv:1901.01543):

1. **2D Laplace Equation**: $\Delta u = u_{xx} + u_{yy} = 0$
   - Infinite-dimensional Cauchy–Riemann symmetry structure
   - The 6-dimensional conformal Lie algebra $\mathfrak{conf}(\mathbb{R}^2) \cong \mathfrak{so}(3,1)$
   - Conformal vector fields and inversion mappings
2. **Conformal Scalar Curvature (Elliptic Liouville Equation)**: $u_{xx} + u_{yy} = K e^u$
   - Preservation of conformal point symmetries
3. **Hyperbolic Liouville Equation**: $u_{tt} - u_{xx} = a e^u$
   - Infinite-dimensional symmetry algebra in light-cone coordinates
   - General analytical solution verification

In [ ]:
import sympy as sp

from symlie import (
    InfinitesimalGenerator,
    adjoint_frechet_derivative,
    frechet_derivative,
    lie_bracket,
    max_derivative_order,
    verify_generator,
)

sp.init_printing()

x, y = sp.symbols("x y")
u = sp.Function("u")(x, y)

# 2D Laplace Equation
laplace_eq = u.diff(x, 2) + u.diff(y, 2)
print("PDE Order:", max_derivative_order(laplace_eq, u, (x, y)))
sp.Eq(laplace_eq, 0)

## 1. Linearization and Determining System of the Laplace Equation

The 2D Laplace operator is formally self-adjoint: $\mathbf{D}^* = \mathbf{D}$.

In [ ]:
Q = sp.Function("Q")(x, y)
D_lap = frechet_derivative(laplace_eq, u, (x, y), Q)
D_star_lap = adjoint_frechet_derivative(laplace_eq, u, (x, y), Q)

print("Fréchet Derivative D(Q):")
display(D_lap)
print("Formal Adjoint D*(Q):")
display(D_star_lap[0])
assert sp.simplify(D_lap[0, 0] - D_star_lap[0]) == 0
print("Confirmed: The Laplace operator is formally self-adjoint!")

## 2. The 6-Dimensional Conformal Lie Algebra $\mathfrak{conf}(\mathbb{R}^2) \cong \mathfrak{so}(3,1)$

The finite-dimensional point symmetries of the Euclidean plane comprise:
- Translations: $\mathbf{p}_x = \partial_x, \quad \mathbf{p}_y = \partial_y$
- Rotation: $\mathbf{j} = -y \partial_x + x \partial_y$
- Dilation: $\mathbf{d} = x \partial_x + y \partial_y$
- Special Conformal Transformations: $\mathbf{c}_x = (x^2 - y^2)\partial_x + 2xy\partial_y, \quad \mathbf{c}_y = 2xy\partial_x + (y^2 - x^2)\partial_y$

In [ ]:
p_x = InfinitesimalGenerator(xi=(1, 0), phi=(0,))
p_y = InfinitesimalGenerator(xi=(0, 1), phi=(0,))
j_rot = InfinitesimalGenerator(xi=(-y, x), phi=(0,))
d_scale = InfinitesimalGenerator(xi=(x, y), phi=(0,))
c_x = InfinitesimalGenerator(xi=(x**2 - y**2, 2 * x * y), phi=(0,))
c_y = InfinitesimalGenerator(xi=(2 * x * y, y**2 - x**2), phi=(0,))

generators = [
    ("p_x", p_x),
    ("p_y", p_y),
    ("j_rot", j_rot),
    ("d_scale", d_scale),
    ("c_x", c_x),
    ("c_y", c_y),
]
for name, gen in generators:
    valid = verify_generator(laplace_eq, u, (x, y), gen)
    print(f"Generator {name:7s} invariant: {valid}")

### Commutation Relations of $\mathfrak{so}(3,1)$

We evaluate key Lie brackets:
$$[\mathbf{p}_x, \mathbf{c}_x] = 2 \mathbf{d}, \quad [\mathbf{p}_x, \mathbf{c}_y] = -2 \mathbf{j}, \quad [\mathbf{d}, \mathbf{c}_x] = \mathbf{c}_x$$

In [ ]:
bracket_px_cx = lie_bracket(p_x, c_x, u, (x, y))
bracket_px_cy = lie_bracket(p_x, c_y, u, (x, y))
bracket_d_cx = lie_bracket(d_scale, c_x, u, (x, y))

print("[p_x, c_x] =", bracket_px_cx)
print("[p_x, c_y] =", bracket_px_cy)
print("[d, c_x]   =", bracket_d_cx)

assert bracket_px_cx.xi == (2 * x, 2 * y)
assert bracket_px_cy.xi == (2 * y, -2 * x)
assert bracket_d_cx.xi == (x**2 - y**2, 2 * x * y)
print("Commutators verified: so(3,1) algebra structure confirmed!")

## 3. The Conformal Scalar Curvature Equation (Elliptic Liouville)

In isothermal coordinates with constant Gaussian curvature $K$:
$$u_{xx} + u_{yy} = K e^u$$

The conformal vector field $\mathbf{c}_x$ extends to the dependent variable via:
$$\mathbf{v}_{\text{conf}} = (x^2 - y^2)\partial_x + 2xy\partial_y - 4x\partial_u$$

In [ ]:
K = sp.symbols("K")
elliptic_liouville = u.diff(x, 2) + u.diff(y, 2) - K * sp.exp(u)

v_conf_liouville = InfinitesimalGenerator(
    xi=(x**2 - y**2, 2 * x * y),
    phi=(-4 * x,),
)

valid_liouv = verify_generator(elliptic_liouville, u, (x, y), v_conf_liouville)
print("Elliptic Liouville Conformal Invariance Verified:", valid_liouv)
assert valid_liouv

## 4. Hyperbolic Liouville Equation and General Analytical Solution

For the hyperbolic Liouville equation in physical space-time coordinates $(t, x)$:
$$u_{tt} - u_{xx} = a e^u$$

The general solution in terms of two arbitrary analytic functions $f(\xi)$ and $g(\eta)$ is:
$$u(t, x) = \ln \left( \frac{8}{a} \frac{f'(t+x)g'(t-x)}{(f(t+x)+g(t-x))^2} \right)$$

In [ ]:
t = sp.symbols("t")
u_tx = sp.Function("u")(x, t)
a = sp.symbols("a", positive=True)
hyp_liouville = u_tx.diff(t, 2) - u_tx.diff(x, 2) - a * sp.exp(u_tx)

# Scaling / conformal generator X = x d/dx + t d/dt - 2 d/du
scale_gen = InfinitesimalGenerator(xi=(x, t), phi=(-2,))
assert verify_generator(hyp_liouville, u_tx, (x, t), scale_gen)
print("Hyperbolic Liouville scaling symmetry verified!")

# Verify the arbitrary-function family directly in light-cone coordinates.
r, s = sp.symbols("r s")
f = sp.Function("f")
g = sp.Function("g")

general_solution_rs = sp.log(8 / a * f(r).diff(r) * g(s).diff(s) / (f(r) + g(s)) ** 2)
# Since r=t+x and s=t-x, u_tt-u_xx = 4*u_rs.
general_residual = sp.simplify(
    4 * general_solution_rs.diff(r, s) - a * sp.exp(general_solution_rs)
)
print("Arbitrary-function solution residual:", general_residual)
assert general_residual == 0
print("General Liouville solution family verified in light-cone coordinates.")